In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

## Trainer
a complete training and evaluation loop for Transformers’ PyTorch models.

**Trainer contains all the necessary components of a training loop.**

-calculate the loss from a training step  
-calculate the gradients with the backward method  
-update the weights based on the gradients  
-repeat until the predetermined number of epochs is reached

In [2]:
import torch
import torch.nn as nn
from transformers import Trainer

2026-02-04 02:19:53.238394: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1770171593.454701      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1770171593.520256      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1770171594.070333      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770171594.070370      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770171594.070373      55 computation_placer.cc:177] computation placer alr

In [3]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

In [4]:
from huggingface_hub import login
login("your HF Token")

### load dataset

In [5]:
from datasets import load_dataset

emotion_ds = load_dataset('emotion')
emotion_ds

README.md: 0.00B [00:00, ?B/s]

split/train-00000-of-00001.parquet:   0%|          | 0.00/1.03M [00:00<?, ?B/s]

split/validation-00000-of-00001.parquet:   0%|          | 0.00/127k [00:00<?, ?B/s]

split/test-00000-of-00001.parquet:   0%|          | 0.00/129k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/16000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 16000
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 2000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 2000
    })
})

In [6]:
emotion_ds['train'][0]

{'text': 'i didnt feel humiliated', 'label': 0}

In [7]:
emotion_df = emotion_ds['train'].to_pandas()
emotion_df.head()

,text,label
0,i didnt feel humiliated,0
1,i can go from feeling so hopeless to so damned...,0
2,im grabbing a minute to post i feel greedy wrong,3
3,i am ever feeling nostalgic about the fireplac...,2
4,i am feeling grouchy,3


### features

In [8]:
features = emotion_ds['train'].features
features

{'text': Value('string'),
 'label': ClassLabel(names=['sadness', 'joy', 'love', 'anger', 'fear', 'surprise'])}

In [9]:
#id to label
id2label = {idx:features['label'].int2str(idx) for idx in range(6)}
id2label

{0: 'sadness', 1: 'joy', 2: 'love', 3: 'anger', 4: 'fear', 5: 'surprise'}

In [10]:
#label to id
label2id = {val : key for key,val in id2label.items()}
label2id

{'sadness': 0, 'joy': 1, 'love': 2, 'anger': 3, 'fear': 4, 'surprise': 5}

### tokenization

In [11]:
from transformers import AutoTokenizer

model_ckpt = "microsoft/MiniLM-L12-H384-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)

tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [12]:
tokenizer(emotion_ds['train']['text'][:1])

{'input_ids': [[101, 1045, 2134, 2102, 2514, 26608, 102]], 'token_type_ids': [[0, 0, 0, 0, 0, 0, 0]], 'attention_mask': [[1, 1, 1, 1, 1, 1, 1]]}

In [13]:
def tokenize_text(examples):
    return tokenizer(examples['text'], truncation=True, max_length=512)

In [14]:
emotion_ds = emotion_ds.map(tokenize_text,batched=True)
emotion_ds

Map:   0%|          | 0/16000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 16000
    })
    validation: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 2000
    })
    test: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 2000
    })
})

## dealing with imbalanced classes

In [15]:
emotion_df['label'].value_counts()

label
1    5362
0    4666
3    2159
4    1937
2    1304
5     572
Name: count, dtype: int64

In [16]:
class_weights = (1-(emotion_df['label'].value_counts().sort_index() / len(emotion_df))).values
class_weights

array([0.708375 , 0.664875 , 0.9185   , 0.8650625, 0.8789375, 0.96425  ])

In [17]:
#convert to tensors
class_weights = torch.from_numpy(class_weights).float().to('cuda')

In [18]:
emotion_ds = emotion_ds.rename_column('label','labels')

In [19]:
class WeightedLossTrainer(Trainer):
    def compute_loss(self,model,inputs,return_outputs=False,**kwargs):
        
        outputs = model(**inputs)
        logits = outputs.get('logits')

        #extract labels
        labels = inputs.get('labels')

        #loss func with class weights
        loss_func = nn.CrossEntropyLoss(weight=class_weights)

        #compute loss
        loss = loss_func(logits, labels)

        return (loss,outputs) if return_outputs else loss

In [20]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    model_ckpt,
    num_labels=6,
    id2label = id2label,
    label2id = label2id
)
model = model.to(device)

pytorch_model.bin:   0%|          | 0.00/133M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at microsoft/MiniLM-L12-H384-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [21]:
from sklearn.metrics import f1_score

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    f1 = f1_score(labels, preds, average='weighted')
    return {'f1':f1}

In [22]:
from transformers import TrainingArguments, TrainerCallback

class LossLoggingCallback(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is not None:
            print(f"Step {state.global_step}: {logs}")

batch_size = 32
output_dir = "miniLM-emotion-detection"

training_args = TrainingArguments(
    output_dir = output_dir,
    num_train_epochs = 10,
    learning_rate = 1e-5,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='epoch',
    logging_strategy='steps',
    logging_steps=200,
    logging_first_step=True,
    fp16=True,
    push_to_hub = True,
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    report_to='none'
)

In [23]:
trainer = WeightedLossTrainer(
    model = model,
    args = training_args,
    compute_metrics=compute_metrics,
    train_dataset = emotion_ds['train'],
    eval_dataset = emotion_ds['validation'],
    tokenizer = tokenizer,
    callbacks=[LossLoggingCallback()]
)

/tmp/ipykernel_55/2225012502.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `WeightedLossTrainer.__init__`. Use `processing_class` instead.
  trainer = WeightedLossTrainer(


In [24]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Epoch,Training Loss,Validation Loss,F1
1,1.570900,1.244890,0.473455
2,1.212400,0.970592,0.642778
3,1.008200,0.800260,0.751740
4,0.744200,0.655286,0.860690
5,0.628500,0.545917,0.871991
6,0.549600,0.477751,0.885797
7,0.490500,0.427743,0.899561
8,0.413100,0.400151,0.906892
9,0.390800,0.384589,0.908745
10,0.377400,0.378643,0.911422


Step 1: {'loss': 1.7926, 'grad_norm': 0.8572613596916199, 'learning_rate': 1e-05, 'epoch': 0.004}
Step 200: {'loss': 1.5709, 'grad_norm': 4.483524799346924, 'learning_rate': 9.204e-06, 'epoch': 0.8}
Step 250: {'eval_loss': 1.2448903322219849, 'eval_f1': 0.4734547914812232, 'eval_runtime': 1.4601, 'eval_samples_per_second': 1369.745, 'eval_steps_per_second': 21.916, 'epoch': 1.0}


/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Step 400: {'loss': 1.2124, 'grad_norm': 6.129154682159424, 'learning_rate': 8.404000000000001e-06, 'epoch': 1.6}
Step 500: {'eval_loss': 0.9705923795700073, 'eval_f1': 0.6427776091226117, 'eval_runtime': 1.4291, 'eval_samples_per_second': 1399.494, 'eval_steps_per_second': 22.392, 'epoch': 2.0}


...

/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Step 2200: {'loss': 0.3908, 'grad_norm': 6.045101642608643, 'learning_rate': 1.204e-06, 'epoch': 8.8}
Step 2250: {'eval_loss': 0.3845886290073395, 'eval_f1': 0.9087454648844453, 'eval_runtime': 1.4983, 'eval_samples_per_second': 1334.826, 'eval_steps_per_second': 21.357, 'epoch': 9.0}


/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Step 2400: {'loss': 0.3774, 'grad_norm': 8.44678783416748, 'learning_rate': 4.04e-07, 'epoch': 9.6}
Step 2500: {'eval_loss': 0.37864261865615845, 'eval_f1': 0.9114218060880537, 'eval_runtime': 1.5131, 'eval_samples_per_second': 1321.808, 'eval_steps_per_second': 21.149, 'epoch': 10.0}
Step 2500: {'train_runtime': 361.8006, 'train_samples_per_second': 442.233, 'train_steps_per_second': 6.91, 'total_flos': 1165199356613376.0, 'train_loss': 0.7106415817260742, 'epoch': 10.0}


TrainOutput(global_step=2500, training_loss=0.7106415817260742, metrics={'train_runtime': 361.8006, 'train_samples_per_second': 442.233, 'train_steps_per_second': 6.91, 'total_flos': 1165199356613376.0, 'train_loss': 0.7106415817260742, 'epoch': 10.0})

In [26]:
from transformers import pipeline

model_ckpt = 'model-checkpoint-where-you-saved-your-model-on-HF'
pipe = pipeline('text-classification',model=model_ckpt)

config.json:   0%|          | 0.00/892 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/133M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

Device set to use cuda:0


In [27]:
pipe("I woke up early but was feeling bit dizzy so i meditated for a while and felt fresh.")

[{'label': 'fear', 'score': 0.7255425453186035}]

In [29]:
sentences = [
    "I woke up early but was feeling bit dizzy so i meditated for a while and felt fresh.",
    "I am extremely happy today because I got selected for the internship!",
    "I feel very sad and lonely these days.",
    "I am angry about how unfair this situation is."
]

for s in sentences:
    out = pipe(s)
    print("\nSentence:", s)
    print("Emotion:", out[0]["label"], "| Score:", round(out[0]["score"], 3))


Sentence: I woke up early but was feeling bit dizzy so i meditated for a while and felt fresh.
Emotion: fear | Score: 0.726

Sentence: I am extremely happy today because I got selected for the internship!
Emotion: joy | Score: 0.918

Sentence: I feel very sad and lonely these days.
Emotion: sadness | Score: 0.927

Sentence: I am angry about how unfair this situation is.
Emotion: anger | Score: 0.532
